In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# keras / tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, Concatenate


In [2]:
file_list = [
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_1.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_2.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_3.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_4.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_5.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_6.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_7.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_8.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_9.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_10.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_11.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_12.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_13.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_14.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_15.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_16.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_17.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_18.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_19.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_20.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_21.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_22.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_23.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_24.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_25.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_26.csv',
    r'C:\Users\franc\Desktop\GitHub\PSD_data\PSD_EEG_Subject_27.csv'
]

print("Number of subject datasets:", len(file_list))


Number of subject datasets: 27


In [3]:
# Select one subject to inspect (0..26)
subject_idx = 0

psd_df = pd.read_csv(file_list[subject_idx])
print("Loaded:", os.path.basename(file_list[subject_idx]))
print(psd_df.shape)

display(psd_df.head())


Loaded: PSD_EEG_Subject_1.csv
(1536, 201)


,PO7_f1,PO7_f2,PO7_f3,PO7_f4,PO7_f5,PO7_f6,PO7_f7,PO7_f8,PO7_f9,PO7_f10,...,PO8_f42,PO8_f43,PO8_f44,PO8_f45,PO8_f46,PO8_f47,PO8_f48,PO8_f49,PO8_f50,Event Id
0,0.082666,0.037937,0.000100,0.000364,0.016201,0.290114,1.587241,2.140813,2.507484,1.207525,...,0.000187,0.000085,0.000056,0.000041,0.000015,0.000006,0.000009,0.000003,0.000004,0.0
1,0.126811,0.058032,0.000147,0.000926,0.006867,0.268436,1.378333,1.744671,1.301250,0.620963,...,0.000224,0.000117,0.000060,0.000024,0.000007,0.000003,0.000006,0.000003,0.000009,0.0
2,0.158416,0.071360,0.000254,0.000916,0.010896,0.164307,0.583307,1.058819,1.247543,0.813192,...,0.000310,0.000128,0.000054,0.000015,0.000005,0.000004,0.000006,0.000002,0.000005,0.0
3,0.108623,0.049146,0.000155,0.000834,0.007435,0.048091,0.315809,0.958991,1.864799,1.088303,...,0.000361,0.000140,0.000050,0.000016,0.000004,0.000005,0.000005,0.000004,0.000003,0.0
4,0.153826,0.070596,0.000168,0.000727,0.010289,0.020010,0.360903,1.493752,2.538120,1.166864,...,0.000276,0.000153,0.000052,0.000013,0.000003,0.000004,0.000004,0.000003,0.000003,0.0


In [4]:
def prepare_data_single_subject(psd_df, random_state=42):
    # Features and labels
    X = psd_df.drop("Event Id", axis=1).astype(float)
    y = psd_df["Event Id"]

    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=0.8, shuffle=True, random_state=random_state, stratify=y
    )

    # Min-Max normalization (fit on train, apply on test)
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build training DataFrame
    train_df = pd.DataFrame(X_train_scaled)
    train_df["Event Id"] = y_train.values

    # Oversampling (same as your code)
    class_counts = train_df["Event Id"].value_counts()
    max_samples = class_counts.max()

    balanced_dfs = []
    for cls, count in class_counts.items():
        df_cls = train_df[train_df["Event Id"] == cls]
        if count < max_samples:
            df_upsampled = resample(df_cls, replace=True, n_samples=max_samples, random_state=random_state)
        else:
            df_upsampled = df_cls
        balanced_dfs.append(df_upsampled)

    df_train_balanced_OS = pd.concat(balanced_dfs)
    df_train_balanced_OS = df_train_balanced_OS.sample(frac=1, random_state=random_state)

    X_train_balanced_OS = df_train_balanced_OS.drop("Event Id", axis=1).values
    y_train_balanced_OS = df_train_balanced_OS["Event Id"].values

    return X_train_balanced_OS, y_train_balanced_OS, X_test_scaled, y_test


In [5]:
# === TEXT RESULTS DIRECTORY ===
base_results_dir = r"C:\Users\franc\Desktop\GitHub\01-Classifier_Selection"
results_txt_dir = os.path.join(base_results_dir, "results")
os.makedirs(results_txt_dir, exist_ok=True)

def print_and_save(text, file):
    print(text)
    file.write(text + "\n")


In [6]:
# === RANDOM FOREST CLASSIFICATION (PER SUBJECT) + SAVE PRINTS ===

acc_list = []
prec_list = []
rec_list = []
f1_list = []

out_txt_path = os.path.join(results_txt_dir, "RF_results.txt")

with open(out_txt_path, "w", encoding="utf-8") as f:

    for file_path in file_list:

        subject_name = os.path.splitext(os.path.basename(file_path))[0]
        print_and_save("\n==============================", f)
        print_and_save(f"RANDOM FOREST - {subject_name}", f)
        print_and_save("==============================", f)

        psd_df = pd.read_csv(file_path)

        X_train_balanced_OS, y_train_balanced_OS, X_test_scaled, y_test = prepare_data_single_subject(psd_df)

        rcf = RandomForestClassifier(
            n_estimators=50,
            random_state=42,
            class_weight='balanced'
        )
        rcf.fit(X_train_balanced_OS, y_train_balanced_OS)

        y_hat_rf = rcf.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_hat_rf)
        print_and_save(f"Model Accuracy: {acc:.3f}", f)

        df_confusion = pd.crosstab(
            pd.Series(y_test, name="Actual"),
            pd.Series(y_hat_rf, name="Predicted")
        )

        print_and_save("\nConfusion Matrix:", f)
        print_and_save(df_confusion.to_string(), f)

        report = classification_report(y_test, y_hat_rf, output_dict=True)
        print_and_save("\nClassification Report:", f)
        print_and_save(classification_report(y_test, y_hat_rf), f)

        acc_list.append(acc)
        prec_list.append(report["macro avg"]["precision"])
        rec_list.append(report["macro avg"]["recall"])
        f1_list.append(report["macro avg"]["f1-score"])

    print_and_save("\n==============================", f)
    print_and_save("MEAN METRICS ACROSS SUBJECTS (RANDOM FOREST)", f)
    print_and_save("==============================", f)
    print_and_save(f"Accuracy  : {np.mean(acc_list):.3f}", f)
    print_and_save(f"Precision : {np.mean(prec_list):.3f}", f)
    print_and_save(f"Recall    : {np.mean(rec_list):.3f}", f)
    print_and_save(f"F1-score  : {np.mean(f1_list):.3f}", f)



RANDOM FOREST - PSD_EEG_Subject_1
Model Accuracy: 0.867

Confusion Matrix:
Predicted  0.0      33024.0  33025.0  33026.0  33027.0
Actual                                                
0.0             14        3        2        4        5
33024.0          3        1        2        2        1
33025.0          8        1        2        0        1
33026.0          5        1        2        1        1
33027.0          2        1        0        0        0

Classification Report:
              precision    recall  f1-score   support

         0.0       0.79      0.95      0.86       128
     33024.0       0.97      0.82      0.89        45
     33025.0       0.97      0.84      0.90        45
     33026.0       0.93      0.84      0.88        45
     33027.0       0.92      0.73      0.81        45

    accuracy                           0.87       308
   macro avg       0.92      0.84      0.87       308
weighted avg       0.88      0.87      0.87       308


RANDOM FOREST - PSD_EEG_S

In [7]:
# === SVM RBF CLASSIFICATION (PER SUBJECT) + SAVE PRINTS ===

acc_list = []
prec_list = []
rec_list = []
f1_list = []

out_txt_path = os.path.join(results_txt_dir, "SVM_results.txt")

with open(out_txt_path, "w", encoding="utf-8") as f:

    for file_path in file_list:

        subject_name = os.path.splitext(os.path.basename(file_path))[0]
        print_and_save("\n==============================", f)
        print_and_save(f"SVM RBF - {subject_name}", f)
        print_and_save("==============================", f)

        psd_df = pd.read_csv(file_path)

        X_train_balanced_OS, y_train_balanced_OS, X_test_scaled, y_test = prepare_data_single_subject(psd_df)

        svm_rbf = SVC(kernel='rbf', class_weight='balanced')
        svm_rbf.fit(X_train_balanced_OS, y_train_balanced_OS)

        y_hat_svm = svm_rbf.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_hat_svm)
        print_and_save(f"Model Accuracy: {acc:.3f}", f)

        df_confusion = pd.crosstab(
            pd.Series(y_test, name="Actual"),
            pd.Series(y_hat_svm, name="Predicted")
        )

        print_and_save("\nConfusion Matrix:", f)
        print_and_save(df_confusion.to_string(), f)

        report = classification_report(y_test, y_hat_svm, output_dict=True)
        print_and_save("\nClassification Report:", f)
        print_and_save(classification_report(y_test, y_hat_svm), f)

        acc_list.append(acc)
        prec_list.append(report["macro avg"]["precision"])
        rec_list.append(report["macro avg"]["recall"])
        f1_list.append(report["macro avg"]["f1-score"])

    print_and_save("\n==============================", f)
    print_and_save("MEAN METRICS ACROSS SUBJECTS (SVM RBF)", f)
    print_and_save("==============================", f)
    print_and_save(f"Accuracy  : {np.mean(acc_list):.3f}", f)
    print_and_save(f"Precision : {np.mean(prec_list):.3f}", f)
    print_and_save(f"Recall    : {np.mean(rec_list):.3f}", f)
    print_and_save(f"F1-score  : {np.mean(f1_list):.3f}", f)



SVM RBF - PSD_EEG_Subject_1
Model Accuracy: 0.925

Confusion Matrix:
Predicted  0.0      33024.0  33025.0  33026.0  33027.0
Actual                                                
0.0             12        3        3        5        5
33024.0          2        1        2        1        3
33025.0          8        1        2        0        1
33026.0          5        1        2        1        1
33027.0          2        1        0        0        0

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.92      0.91       128
     33024.0       0.98      0.93      0.95        45
     33025.0       0.95      0.91      0.93        45
     33026.0       0.93      0.93      0.93        45
     33027.0       0.91      0.93      0.92        45

    accuracy                           0.93       308
   macro avg       0.94      0.93      0.93       308
weighted avg       0.93      0.93      0.93       308


SVM RBF - PSD_EEG_Subject_2
Mod

In [8]:
# === MLP NEURAL NETWORK ===

# === DIRECTORY FOR BEST MLP MODELS ===
base_dir = r"C:\Users\franc\Desktop\GitHub\01-Classifier_Selection"
best_model_dir = os.path.join(base_dir, "best_model_mlp")
os.makedirs(best_model_dir, exist_ok=True)

acc_list = []
prec_list = []
rec_list = []
f1_list = []

out_txt_path = os.path.join(results_txt_dir, "MLP_results.txt")

with open(out_txt_path, "w", encoding="utf-8") as f:

    for file_path in file_list:

        subject_name = os.path.splitext(os.path.basename(file_path))[0]
        print_and_save("\n==============================", f)
        print_and_save(f"MLP - {subject_name}", f)
        print_and_save("==============================", f)

        psd_df = pd.read_csv(file_path)

        X_train_balanced_OS, y_train_balanced_OS, X_test_scaled, y_test = prepare_data_single_subject(psd_df)

        classes = np.sort(np.unique(np.concatenate([y_train_balanced_OS, y_test])))
        to_index = {c: i for i, c in enumerate(classes)}

        y_train_cat = to_categorical(
            [to_index[c] for c in y_train_balanced_OS],
            num_classes=len(classes)
        )

        checkpoint = ModelCheckpoint(
            filepath=os.path.join(best_model_dir, f"best_model_{subject_name}.keras"),
            monitor='val_loss',
            save_best_only=True,
            mode='min',
            verbose=0
        )

        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )

        model = Sequential()
        model.add(InputLayer(shape=(X_train_balanced_OS.shape[1],)))
        model.add(Dense(256, activation='relu'))
        model.add(Dense(128, activation='relu'))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(len(classes), activation='softmax'))

        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        model.fit(
            X_train_balanced_OS,
            y_train_cat,
            validation_split=0.2,
            epochs=200,
            batch_size=32,
            callbacks=[checkpoint, reduce_lr],
            verbose=0
        )

        y_hat_nn = model.predict(X_test_scaled, verbose=0)
        y_hat_nn_labels = classes[np.argmax(y_hat_nn, axis=1)]

        acc = accuracy_score(y_test, y_hat_nn_labels)
        print_and_save(f"Model Accuracy: {acc:.3f}", f)

        df_confusion = pd.crosstab(
            pd.Series(y_test, name="Actual"),
            pd.Series(y_hat_nn_labels, name="Predicted")
        )

        print_and_save("\nConfusion Matrix:", f)
        print_and_save(df_confusion.to_string(), f)

        report = classification_report(y_test, y_hat_nn_labels, output_dict=True)
        print_and_save("\nClassification Report:", f)
        print_and_save(classification_report(y_test, y_hat_nn_labels), f)

        acc_list.append(acc)
        prec_list.append(report["macro avg"]["precision"])
        rec_list.append(report["macro avg"]["recall"])
        f1_list.append(report["macro avg"]["f1-score"])

    print_and_save("\n==============================", f)
    print_and_save("MEAN METRICS ACROSS SUBJECTS (MLP)", f)
    print_and_save("==============================", f)
    print_and_save(f"Accuracy  : {np.mean(acc_list):.3f}", f)
    print_and_save(f"Precision : {np.mean(prec_list):.3f}", f)
    print_and_save(f"Recall    : {np.mean(rec_list):.3f}", f)
    print_and_save(f"F1-score  : {np.mean(f1_list):.3f}", f)



MLP - PSD_EEG_Subject_1
Model Accuracy: 0.922

Confusion Matrix:
Predicted  0.0      33024.0  33025.0  33026.0  33027.0
Actual                                                
0.0             14        2        3        5        4
33024.0          1        2        2        1        3
33025.0          8        1        2        0        1
33026.0          3        1        3        1        2
33027.0          2        1        0        0        0

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.92      0.91       128
     33024.0       1.00      0.91      0.95        45
     33025.0       0.93      0.91      0.92        45
     33026.0       0.91      0.93      0.92        45
     33027.0       0.91      0.93      0.92        45

    accuracy                           0.92       308
   macro avg       0.93      0.92      0.93       308
weighted avg       0.92      0.92      0.92       308


MLP - PSD_EEG_Subject_2
Model Accur

In [10]:
# === RESIDUAL NEURAL NETWORK === #


# === DIRECTORY FOR BEST RESNET WEIGHTS ===
best_resnet_dir = os.path.join(base_dir, "best_model_resnet")
os.makedirs(best_resnet_dir, exist_ok=True)


# ======= RESNET DEFINITION ======= #
def create_resnet(input_dim, num_classes, dropout_pcg=0.25, factors=256):
    inp = Input(shape=(input_dim,))
    out = Dense(factors, activation="relu")(inp)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    res = Dense(factors, activation="relu")(out)
    res = BatchNormalization()(res)
    res = Dropout(dropout_pcg)(res)

    out = Add()([out, res])
    out = Concatenate()([out, inp])
    out = Dense(num_classes, activation="softmax")(out)

    m = Model(inp, out)
    m.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.RMSprop(1e-3),
        metrics=["accuracy"]
    )
    return m



acc_list = []
prec_list = []
rec_list = []
f1_list = []

out_txt_path = os.path.join(results_txt_dir, "RESNET_results.txt")

with open(out_txt_path, "w", encoding="utf-8") as f:

    for file_path in file_list:

        subject_name = os.path.splitext(os.path.basename(file_path))[0]
        print_and_save("\n==============================", f)
        print_and_save(f"RESNET - {subject_name}", f)
        print_and_save("==============================", f)

        # ======= LOAD SUBJECT DATASET ======= #
        df = pd.read_csv(file_path)

        X = df.drop("Event Id", axis=1).astype(float).values
        y = df["Event Id"].values

        input_dim = X.shape[1]

        # Classes mapping (consistent for this subject)
        classes = np.array(sorted(np.unique(y)))
        num_classes = len(classes)
        to_index = {c: i for i, c in enumerate(classes)}

        # ======= TRAIN/TEST SPLIT ======= #
        Xtr, Xte, ytr, yte = train_test_split(
            X, y,
            train_size=0.8,
            shuffle=True,
            random_state=42,
            stratify=y
        )

        # ======= SCALER (fit on train only) ======= #
        scaler = MinMaxScaler().fit(Xtr)
        Xtr_s = scaler.transform(Xtr)
        Xte_s = scaler.transform(Xte)

        # ======= OVERSAMPLING (train only) ======= #
        df_tr = pd.DataFrame(Xtr_s)
        df_tr["Event Id"] = ytr

        max_n = df_tr["Event Id"].value_counts().max()
        dfs = []
        for cls, sub in df_tr.groupby("Event Id"):
            if len(sub) < max_n:
                sub = resample(sub, replace=True, n_samples=max_n, random_state=42)
            dfs.append(sub)

        df_bal = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
        Xtr_bal = df_bal.drop("Event Id", axis=1).values
        ytr_bal = df_bal["Event Id"].values

        # ======= ONE-HOT ======= #
        ytr_cat = to_categorical([to_index[c] for c in ytr_bal], num_classes=num_classes)
        yte_idx = np.array([to_index[c] for c in yte])

        # ======= TRAINING ======= #
        model = create_resnet(input_dim, num_classes)

        ckpt_path = os.path.join(best_resnet_dir, f"best_resnet_{subject_name}.weights.h5")
        ckpt = ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            mode="min",
            verbose=0
        )
        rlr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )

        model.fit(
            Xtr_bal, ytr_cat,
            validation_split=0.2,
            epochs=150,
            batch_size=64,
            callbacks=[ckpt, rlr],
            verbose=0
        )

        # ======= TEST ======= #
        model.load_weights(ckpt_path)
        ypred = model.predict(Xte_s, verbose=0)
        ypred_lab = np.argmax(ypred, axis=1)

        acc = accuracy_score(yte_idx, ypred_lab)
        print_and_save(f"Test Accuracy: {acc:.4f}", f)

        # Print & save report (using indexed labels + readable names)
        rep_str = classification_report(
            yte_idx, ypred_lab,
            target_names=[str(c) for c in classes],
            zero_division=0
        )
        print_and_save("\nClassification Report:", f)
        print_and_save(rep_str, f)

        # Store metrics (macro avg) for final mean (optional, like your others)
        rep_dict = classification_report(
            yte_idx, ypred_lab,
            target_names=[str(c) for c in classes],
            output_dict=True,
            zero_division=0
        )

        acc_list.append(acc)
        prec_list.append(rep_dict["macro avg"]["precision"])
        rec_list.append(rep_dict["macro avg"]["recall"])
        f1_list.append(rep_dict["macro avg"]["f1-score"])

    # ======= MEAN METRICS ======= #
    print_and_save("\n==============================", f)
    print_and_save("MEAN METRICS ACROSS SUBJECTS (RESNET)", f)
    print_and_save("==============================", f)
    print_and_save(f"Accuracy  : {np.mean(acc_list):.3f}", f)
    print_and_save(f"Precision : {np.mean(prec_list):.3f}", f)
    print_and_save(f"Recall    : {np.mean(rec_list):.3f}", f)
    print_and_save(f"F1-score  : {np.mean(f1_list):.3f}", f)



RESNET - PSD_EEG_Subject_1
Test Accuracy: 0.9286

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.91      0.92       128
     33024.0       0.98      0.93      0.95        45
     33025.0       0.93      0.93      0.93        45
     33026.0       0.93      0.96      0.95        45
     33027.0       0.88      0.93      0.90        45

    accuracy                           0.93       308
   macro avg       0.93      0.93      0.93       308
weighted avg       0.93      0.93      0.93       308


RESNET - PSD_EEG_Subject_2
Test Accuracy: 0.9383

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94       128
     33024.0       1.00      0.98      0.99        45
     33025.0       0.93      0.93      0.93        45
     33026.0       0.93      0.89      0.91        45
     33027.0       0.88      0.96      0.91        45

    accuracy                         